In [1]:
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
euro_govs = pd.read_csv('gov_fut.csv')

In [3]:
euro_govs_isin = tuple(euro_govs['isin'].unique())

In [ ]:
# sample: Sep 2024 Bund future (FGBL) on one trading day
sample_isin = 'DE000F0FSQN3'
sample_date = '2024-07-15'

In [ ]:
# step 1: identifiers and counterparties (buyer / seller instead of reporter / other / direction)
query = f"""
SELECT
    e.tec_ruti,
    e.reference_period,
    CASE WHEN e.leg1_direction = 'BYER' THEN e.leg1_reporting_cpty_id ELSE e.leg1_other_cpty_id END AS buyer_id,
    CASE WHEN e.leg1_direction = 'BYER' THEN s_rep.sector              ELSE s_oth.sector              END AS buyer_sector,
    CASE WHEN e.leg1_direction = 'SLLR' THEN e.leg1_reporting_cpty_id ELSE e.leg1_other_cpty_id END AS seller_id,
    CASE WHEN e.leg1_direction = 'SLLR' THEN s_rep.sector              ELSE s_oth.sector              END AS seller_sector
FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_rep
    ON e.leg1_reporting_cpty_id = s_rep.lei
LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_oth
    ON e.leg1_other_cpty_id = s_oth.lei
WHERE e.reference_period = '{sample_date}'
    AND e.leg1_product_isin = '{sample_isin}'
    AND e.leg1_direction IN ('BYER', 'SLLR')
"""

df = pd.read_sql_query(query, cnxn)
print(df.shape)
df.head()